[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/finetuning/blob/main/chapter_07/listing_7.1-7.6.ipynb)

In [1]:
import sys
if "google.colab" in sys.modules:
    !pip install -q -U unsloth transformers peft trl datasets bitsandbytes accelerate

### Listing 7.1: Setting up the Environment

In [2]:
from unsloth import FastLanguageModel, PatchDPOTrainer
import os
import torch
import warnings
from datasets import load_dataset
from transformers import AutoTokenizer
from trl import DPOTrainer, DPOConfig

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if torch.cuda.is_available():
   device = "cuda"
elif torch.backends.mps.is_available():
   device = "mps"
else:
   device = "cpu"

if device == "cuda" and torch.cuda.get_device_capability()[0] >= 8:
   compute_dtype = torch.bfloat16
else:
   compute_dtype = torch.float16
print(f"Using device: {device} | Dtype: {compute_dtype}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


INFO 08-23 13:39:06 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.


WARNING 08-23 13:39:06 [nixl_utils.py:34] NIXL is not available


WARNING 08-23 13:39:06 [nixl_utils.py:44] NIXL agent config is not available


🦥 Unsloth Zoo will now patch everything to make training faster!


Using device: cuda | Dtype: torch.bfloat16


### Listing 7.2: Loading Model, Tokenizer, and Configuring LoRA

In [3]:
MODEL_ID = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
   model_name=MODEL_ID,
   max_seq_length=1024,
   dtype=None,
   load_in_4bit=True,
)
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
   tokenizer,
   chat_template="chatml",
)
model = FastLanguageModel.get_peft_model(
   model,
   r=16,
   lora_alpha=16,
   target_modules=[
       "q_proj", "k_proj", "v_proj",
       "o_proj", "gate_proj", "up_proj", "down_proj",
   ],
   lora_dropout=0,
   bias="none",
   use_gradient_checkpointing="unsloth",
   random_state=42,
)

==((====))==  Unsloth 2026.8.1: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.20.2.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


[unsloth.chat_templates|WARNING]Unsloth: Will map <|im_end|> to EOS = <|im_end|>.


Unsloth 2026.8.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


### Listing 7.3: Preprocessing the Preference Dataset

In [4]:
dataset = load_dataset("argilla/distilabel-intel-orca-dpo-pairs", split="train")
shuffled = dataset.shuffle(seed=42)
train_ds = shuffled.select(range(250))
eval_ds = shuffled.select(range(250, 300))

def format_dpo_example(example):
   messages = [{"role": "user", "content": example["input"]}]
   prompt_str = tokenizer.apply_chat_template(
       messages, tokenize=False, add_generation_prompt=True
   )
   chosen_str = example["chosen"].strip()
   rejected_str = example["rejected"].strip()
   if not chosen_str.endswith("<|im_end|>"):
       chosen_str += "<|im_end|>\n"
   if not rejected_str.endswith("<|im_end|>"):
       rejected_str += "<|im_end|>\n"
   return {
       "prompt": prompt_str,
       "chosen": chosen_str,
       "rejected": rejected_str,
   }

train_mapped = train_ds.map(format_dpo_example, remove_columns=train_ds.column_names)
eval_mapped = eval_ds.map(format_dpo_example, remove_columns=eval_ds.column_names)
print("Prompt Preview:\n", train_mapped[0]["prompt"])
print("Chosen Preview:\n", train_mapped[0]["chosen"])
print("Rejected Preview:\n", train_mapped[0]["rejected"])

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Prompt Preview:
 <|im_start|>user
This is some data: CBS PLAY-BY-PLAY Chris Schenkel (first half) and Ray Scott (second half); 1962 NETWORK CBS.

Generate a detailed description of this data<|im_end|>
<|im_start|>assistant

Chosen Preview:
 Okay, imagine you are watching a fun game on TV with your family. In this case, the game happened in 1962. Now, on TV, there are people who talk to us and tell us what is happening in the game. They help us understand the game better, just like how I'm helping you understand things right now.

In this data, there are two people who talked about the game in 1962. The first person, Chris Schenkel, talked about the game in the first half. The second person, Ray Scott, talked about the game in the second half. Both of them worked for a big TV company called CBS. So, this sentence is just telling us who talked about the game on TV and when they did it.<|im_end|>

Rejected Preview:
 OH MY GOSH, YOU WANT TO KNOW ABOUT THIS SUPER COOL DATA?! 😍

Okay, so let

### Listing 7.4: Generating Pre-DPO Baseline Responses

In [ ]:
FastLanguageModel.for_inference(model)
test_prompts = [
   "Explain why the sky is blue in one concise sentence.",
   "What is a metric ton?",
   "Why did the Roman Empire fall? Explain the primary contributing factors.",
   "What is the difference between existentialism and nihilism?",
]
model.generation_config.max_length = None  # Suppress the max_length warning
base_responses = {}
for prompt in test_prompts:
   messages = [{"role": "user", "content": prompt}]
   inputs = tokenizer.apply_chat_template(
       messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
   )
   inputs = {k: v.to(device) for k, v in inputs.items()}
   with torch.no_grad():
       outputs = model.generate(
           **inputs,
           max_new_tokens=256,
           use_cache=True,
           pad_token_id=tokenizer.eos_token_id,
       )
   base_responses[prompt] = tokenizer.decode(
       outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
   ).strip()
   print(f"Prompt: {prompt}\nBase Response: {base_responses[prompt]}\n" + "-" * 50)

model = FastLanguageModel.for_training(model)

Prompt: Explain why the sky is blue in one concise sentence.
Base Response: The sky appears blue because Earth's atmosphere scatters sunlight's blue wavelengths more than other colors, making them appear brightest and thus creating the blue color we see during the day.
--------------------------------------------------


Prompt: What is a metric ton?
Base Response: A metric ton, also known as a tonne in some contexts, is a unit of measurement for mass. It is defined as exactly 1,000 kilograms (kg). This is equivalent to approximately 2,204.6 pounds.

In everyday use, it's often used for measuring the weight of heavy objects or large quantities of materials. For example, it's commonly used in agriculture to measure the weight of livestock, and in shipping to calculate the weight of goods that need to be transported.
--------------------------------------------------


Prompt: Why did the Roman Empire fall? Explain the primary contributing factors.
Base Response: The fall of the Roman Empire is a subject of extensive historical debate, and it's generally accepted that it was the result of a combination of various factors rather than a single cause. Here are some of the key contributing factors:

1. **Military Overstretch**: The Roman Empire had an extensive military presence throughout its territories, which required large numbers of soldiers to defend against invasions. This led to the constant need for recruiting new soldiers, which put significant strain on the economy.

2. **Economic Problems**: The empire faced several economic challenges, including inflation, over-reliance on agriculture, and a lack of diversification in the economy. Additionally, the cost of maintaining a vast military force drained the treasury.

3. **Political Instability**: The empire saw numerous civil wars and usurpations, which weakened the central authority. Emperors of

Prompt: What is the difference between existentialism and nihilism?
Base Response: Existentialism and nihilism are related philosophical positions, but they have distinct differences:

1. **Existentialism**:
   - It focuses on human existence, freedom, and choice.
   - It often emphasizes the search for meaning in life.
   - Key figures include Jean-Paul Sartre and Martin Heidegger.
   - It can be seen as a form of hope or optimism about the human condition.

2. **Nihilism**:
   - It questions the meaning and value of existence.
   - It often leads to skepticism and despair.
   - It suggests that life is ultimately without inherent meaning or purpose.
   - Notable philosophers include Friedrich Nietzsche and Jean-Paul Sartre (who also contributed to existentialism).
   - Nihilism can lead to a sense of disillusionment and apathy.

In summary, while existentialism acknowledges the possibility of finding meaning in life, it often involves a positive outlook. Nihilism, on the other hand, 

### Listing 7.5: DPO Trainer Setup and Training Execution

In [6]:
PatchDPOTrainer()

training_args = DPOConfig(
   output_dir="qwen2.5-3b-dpo-output",
   beta=0.1,
   max_length=1024,
   per_device_train_batch_size=2,
   gradient_accumulation_steps=4,
   learning_rate=5e-5,
   max_steps=120,
   lr_scheduler_type="cosine",
   warmup_ratio=0.1,
   bf16=(compute_dtype == torch.bfloat16),
   fp16=(compute_dtype == torch.float16),
   logging_steps=10,
   eval_strategy="steps",
   eval_steps=20,
   save_steps=20,
   report_to="none",
)
trainer = DPOTrainer(
   model=model,
   ref_model=None,
   args=training_args,
   train_dataset=train_mapped,
   eval_dataset=eval_mapped,
   processing_class=tokenizer,
)
model.config.use_cache = False

trainer.train()
model.save_pretrained("qwen2.5-3b-dpo-adapter")
tokenizer.save_pretrained("qwen2.5-3b-dpo-adapter")

Extracting prompt in train dataset (num_proc=24):   0%|          | 0/250 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=24):   0%|          | 0/250 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=24):   0%|          | 0/250 [00:00<?, ? examples/s]

Extracting prompt in eval dataset (num_proc=24):   0%|          | 0/50 [00:00<?, ? examples/s]

Applying chat template to eval dataset (num_proc=24):   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=24):   0%|          | 0/50 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 250 | Num Epochs = 4 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
20,0.562800,0.378671,-0.601249,-2.043742,0.884615,1.442492,-223.404221,-300.763306,-1.883741,-1.719044
40,0.239900,0.401040,-0.823670,-3.732280,0.884615,2.908611,-225.628433,-317.648743,-1.799351,-1.657312
60,0.267200,0.431065,-0.725318,-3.015058,0.826923,2.289740,-224.644913,-310.476501,-1.745929,-1.579653
80,0.060600,0.474912,-1.101598,-4.183678,0.846154,3.082080,-228.407700,-322.162659,-1.809981,-1.646143
100,0.068800,0.526281,-1.582945,-5.597703,0.865385,4.014758,-233.221191,-336.302917,-1.869015,-1.713214
120,0.027100,0.516076,-1.640353,-5.814344,0.865385,4.173991,-233.795258,-338.469360,-1.877547,-1.726043


('qwen2.5-3b-dpo-adapter/tokenizer_config.json',
 'qwen2.5-3b-dpo-adapter/special_tokens_map.json',
 'qwen2.5-3b-dpo-adapter/chat_template.jinja',
 'qwen2.5-3b-dpo-adapter/vocab.json',
 'qwen2.5-3b-dpo-adapter/merges.txt',
 'qwen2.5-3b-dpo-adapter/added_tokens.json',
 'qwen2.5-3b-dpo-adapter/tokenizer.json')

### Listing 7.6: Post-DPO Model Evaluation and Comparison

In [ ]:
FastLanguageModel.for_inference(model)

for prompt in test_prompts:
   messages = [{"role": "user", "content": prompt}]
   inputs = tokenizer.apply_chat_template(
       messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
   )
   inputs = {k: v.to(device) for k, v in inputs.items()}

   with torch.no_grad():
       outputs = model.generate(
           **inputs,
           max_new_tokens=256,
           use_cache=True,
           pad_token_id=tokenizer.eos_token_id,
       )

   dpo_response = tokenizer.decode(
       outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
   ).strip()

   print(f"Prompt: {prompt}")
   print(f"Before DPO (Base Model):{base_responses[prompt]}")
   print(f"\nAfter DPO (Aligned Model):{dpo_response}")
   print("-" * 80 + "\n")

Prompt: Explain why the sky is blue in one concise sentence.
Before DPO (Base Model):The sky appears blue because Earth's atmosphere scatters sunlight's blue wavelengths more than other colors, making them appear brightest and thus creating the blue color we see during the day.

After DPO (Aligned Model):The sky appears blue because molecules in Earth's atmosphere scatter sunlight more efficiently in the blue part of the spectrum, causing us to see a predominantly blue sky.
--------------------------------------------------------------------------------



Prompt: What is a metric ton?
Before DPO (Base Model):A metric ton, also known as a tonne in some contexts, is a unit of measurement for mass. It is defined as exactly 1,000 kilograms (kg). This is equivalent to approximately 2,204.6 pounds.

In everyday use, it's often used for measuring the weight of heavy objects or large quantities of materials. For example, it's commonly used in agriculture to measure the weight of livestock, and in shipping to calculate the weight of goods that need to be transported.

After DPO (Aligned Model):A metric ton, also known as a tonne in some countries, is a unit of measurement for mass. It is defined as exactly 1,000 kilograms (kg). This unit is used extensively in many parts of the world for measuring the mass of heavy goods, construction materials, and other large quantities. In some contexts, it can be abbreviated as "t."
--------------------------------------------------------------------------------



Prompt: Why did the Roman Empire fall? Explain the primary contributing factors.
Before DPO (Base Model):The fall of the Roman Empire is a subject of extensive historical debate, and it's generally accepted that it was the result of a combination of various factors rather than a single cause. Here are some of the key contributing factors:

1. **Military Overstretch**: The Roman Empire had an extensive military presence throughout its territories, which required large numbers of soldiers to defend against invasions. This led to the constant need for recruiting new soldiers, which put significant strain on the economy.

2. **Economic Problems**: The empire faced several economic challenges, including inflation, over-reliance on agriculture, and a lack of diversification in the economy. Additionally, the cost of maintaining a vast military force drained the treasury.

3. **Political Instability**: The empire saw numerous civil wars and usurpations, which weakened the central authority. Em

Prompt: What is the difference between existentialism and nihilism?
Before DPO (Base Model):Existentialism and nihilism are related philosophical positions, but they have distinct differences:

1. **Existentialism**:
   - It focuses on human existence, freedom, and choice.
   - It often emphasizes the search for meaning in life.
   - Key figures include Jean-Paul Sartre and Martin Heidegger.
   - It can be seen as a form of hope or optimism about the human condition.

2. **Nihilism**:
   - It questions the meaning and value of existence.
   - It often leads to skepticism and despair.
   - It suggests that life is ultimately without inherent meaning or purpose.
   - Notable philosophers include Friedrich Nietzsche and Jean-Paul Sartre (who also contributed to existentialism).
   - Nihilism can lead to a sense of disillusionment and apathy.

In summary, while existentialism acknowledges the possibility of finding meaning in life, it often involves a positive outlook. Nihilism, on the oth